<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [أرسيني كرافشينكو](http://arseny.info/pages/about-me.html). ترجمة وتحرير [كريستينا بوتسكو](https://www.linkedin.com/in/christinabutsko/)، [يوري كاشنيتسكي](https://yorko.github.io/)، [إيجور بولوسماك](https://www.linkedin.com/in/egor-polusmak/)، [أناستازيا مانوخينا](https://www.linkedin.com/in/anastasiamanokhina/)، [آنا لاريونوفا](https://www.linkedin.com/in/anna-larionova-74434689/)، [يفغيني سوشكو](https://www.linkedin.com/in/evgenysushko/) و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center> الموضوع 6. هندسة الميزات واختيار الميزات</center>
في هذه الدورة، رأينا بالفعل العديد من خوارزميات التعلم الآلي الرئيسية. ومع ذلك، قبل الانتقال إلى الخيارات الأكثر روعة، نود أن نأخذ منعطفًا صغيرًا ونتحدث عن إعداد البيانات. المفهوم المعروف "القمامة في   -   القمامة خارج" ينطبق بنسبة 100٪ على أي مهمة في التعلم الآلي. يمكن لأي محترف ذي خبرة أن يتذكر عدة مرات عندما ثبت أن نموذجًا بسيطًا تم تدريبه على بيانات عالية الجودة أفضل من مجموعة معقدة متعددة النماذج مبنية على بيانات لم تكن نظيفة.
في البداية، أردت مراجعة ثلاث مهام متشابهة ولكنها مختلفة:
* **استخلاص الميزات** و**هندسة الميزات**: تحويل البيانات الأولية إلى ميزات مناسبة للنمذجة؛
* **تحويل الميزة**: تحويل البيانات لتحسين دقة الخوارزمية؛
* **اختيار الميزة**: إزالة الميزات غير الضرورية.لن تحتوي هذه المقالة على أي عمليات حسابية تقريبًا، ولكن سيكون هناك قدر لا بأس به من التعليمات البرمجية. ستستخدم بعض الأمثلة مجموعة البيانات من شركة Renthop، والتي يتم استخدامها في [Two Sigma Connect: Rental Listing Inquiries Kaggle المنافسة](https://www.kaggle.com/c/two-sigma-connect-rental-listing-inquiries). الملف `train.json` يتم الاحتفاظ به أيضًا [هنا](https://drive.google.com/open?id=1_lqydkMrmyNAgG4vU4wVmp6-j7tV0XI8) كـ `renthop_train.json.gz` (لذا قم بفك ضغطه أولاً). في هذه المهمة، تحتاج إلى التنبؤ بمدى شعبية قائمة الإيجار الجديدة، أي تصنيف القائمة إلى ثلاث فئات: `['low', 'medium' , 'high']`. لتقييم الحلول، سوف نستخدم مقياس فقدان السجل (كلما كان أصغر، كان ذلك أفضل). أولئك الذين ليس لديهم حساب Kaggle، سيتعين عليهم التسجيل؛ ستحتاج أيضًا إلى قبول قواعد المسابقة حتى تتمكن من تنزيل البيانات.


In [ ]:
# preload dataset automatically, if not already in place.
import os

import requests

url = "https://drive.google.com/uc?export=download&id=1_lqydkMrmyNAgG4vU4wVmp6-j7tV0XI8"
file_name = "../../data/renthop_train.json.gz"


def load_renthop_dataset(url, target, overwrite=False):
    # check if exists already
    if os.path.isfile(target) and not overwrite:
        print("Dataset is already in place")
        return

    print("Will download the dataset from", url)

    response = requests.get(url)
    open(target, "wb").write(response.content)


load_renthop_dataset(url, file_name)

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_json(file_name, compression="gzip")


## الخطوط العريضة للمادة
1. استخراج الميزة
        1. النصوص
        2. الصور
        3. البيانات الجغرافية المكانية
        4. التاريخ والوقت
        5. السلاسل الزمنية، الويب، إلخ.
2. ميزة التحولات
        1. التطبيع وتغيير التوزيع
        2. التفاعلات
        3. ملء القيم المفقودة
3. اختيار الميزة
        1. الأساليب الإحصائية
        2. الاختيار عن طريق النمذجة
        3. بحث الشبكة



## استخراج الميزة
ومن الناحية العملية، نادراً ما تأتي البيانات في شكل مصفوفات جاهزة للاستخدام. ولهذا السبب تبدأ كل مهمة باستخراج الميزات. في بعض الأحيان، قد يكفي قراءة ملف CSV وتحويله إلى `numpy.array`، ولكن هذا استثناء نادر. دعونا نلقي نظرة على بعض أنواع البيانات الشائعة التي يمكن استخراج الميزات منها.



### النصوص
النص هو نوع من البيانات التي يمكن أن تأتي بتنسيقات مختلفة؛ هناك العديد من أساليب معالجة النصوص التي لا يمكن وضعها في مقالة واحدة. ومع ذلك، سوف نقوم بمراجعة الأكثر شعبية.قبل العمل مع النص، يجب على المرء أن يقوم بترميزه. يتضمن الرمز المميز تقسيم النص إلى وحدات (وبالتالي الرموز المميزة). بكل بساطة، الرموز هي مجرد كلمات. لكن التقسيم حسب الكلمة يمكن أن يفقد بعضًا من المعنى - "سانتا باربرا" هي رمز واحد، وليس اثنين، ولكن لا ينبغي تقسيم "روك أند رول" إلى رمزين. هناك أدوات رمزية جاهزة للاستخدام تأخذ في الاعتبار خصوصيات اللغة، ولكنها ترتكب أخطاء أيضًا، خاصة عند العمل مع مصادر محددة للنص (الصحف، اللغة العامية، الأخطاء الإملائية، الأخطاء المطبعية).
بعد الترميز، ستقوم بتطبيع البيانات. بالنسبة للنص، يتعلق الأمر بالاستئصال و/أو التجسيد؛ هذه عمليات مشابهة تستخدم لمعالجة أشكال مختلفة من الكلمة. يمكن للمرء أن يقرأ عن الفرق بينهما [هنا](http://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html).
والآن بعد أن حولنا الوثيقة إلى سلسلة من الكلمات، يمكننا تمثيلها بالمتجهات. الطريقة الأسهل تسمى حقيبة الكلمات: نقوم بإنشاء متجه بطول المفردات، وحساب عدد تكرارات كل كلمة في النص، ووضع هذا العدد من التكرارات في الموضع المناسب في المتجه. تبدو العملية الموضحة أبسط في الكود:


In [ ]:
texts = ["i have a cat", "you have a dog", "you and i have a cat and a dog"]

vocabulary = list(
    enumerate(set([word for sentence in texts for word in sentence.split()]))
)
print("Vocabulary:", vocabulary)


def vectorize(text):
    vector = np.zeros(len(vocabulary))
    for i, word in vocabulary:
        num = 0
        for w in text:
            if w == word:
                num += 1
        if num:
            vector[i] = num
    return vector


print("Vectors:")
for sentence in texts:
    print(vectorize(sentence.split()))


فيما يلي رسم توضيحي للعملية:
<img src='../../img/bag_of_words.png' width=50%>
هذا تنفيذ ساذج للغاية. من الناحية العملية، تحتاج إلى مراعاة الكلمات المتوقفة، والحد الأقصى لطول المفردات، وهياكل البيانات الأكثر كفاءة (عادةً ما يتم تحويل البيانات النصية إلى ناقل متفرق)، وما إلى ذلك.عند استخدام خوارزميات مثل Bag of Words، نفقد ترتيب الكلمات في النص، مما يعني أن النصين "ليس لدي أبقار" و"لا، لدي أبقار" سيظهران متطابقين بعد التحويل بينما في الواقع، لهما معنى معاكس. لتجنب هذه المشكلة، يمكننا إعادة النظر في خطوة الترميز واستخدام N-grams (*التسلسل* لعدد N من الرموز المميزة المتتالية) بدلاً من ذلك.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(ngram_range=(1, 1))
vect.fit_transform(["no i have cows", "i have no cows"]).toarray()

In [ ]:
vect.vocabulary_

In [ ]:
vect = CountVectorizer(ngram_range=(1, 2))
vect.fit_transform(["no i have cows", "i have no cows"]).toarray()

In [ ]:
vect.vocabulary_


لاحظ أيضًا أنه ليس من الضروري استخدام الكلمات فقط. في بعض الحالات، من الممكن إنشاء N-grams من الأحرف. سيكون هذا النهج قادرًا على مراعاة تشابه الكلمات ذات الصلة أو التعامل مع الأخطاء المطبعية.


In [ ]:
from scipy.spatial.distance import euclidean
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(ngram_range=(3, 3), analyzer="char_wb")

n1, n2, n3, n4 = vect.fit_transform(
    ["andersen", "petersen", "petrov", "smith"]
).toarray()

euclidean(n1, n2), euclidean(n2, n3), euclidean(n3, n4)


الإضافة إلى فكرة حقيبة الكلمات: الكلمات التي نادرًا ما توجد في المجموعة (في جميع المستندات الخاصة بمجموعة البيانات هذه) ولكنها موجودة في هذه الوثيقة بالذات قد تكون أكثر أهمية. ومن ثم فمن المنطقي زيادة وزن الكلمات الخاصة بالمجال لفصلها عن الكلمات الشائعة. يُطلق على هذا الأسلوب اسم TF-IDF (مصطلح تردد الوثيقة العكسي)، والذي لا يمكن كتابته في بضعة أسطر، لذا يجب عليك الاطلاع على التفاصيل في مراجع مثل [هذا الويكي](https://en.wikipedia.org/wiki/Tf%E2%80%93idf). الخيار الافتراضي هو كما يلي:
$$ \large idf(t,D) = \log\frac{\mid D\mid}{df(d,t)+1} $$
$$ \large tfidf(t,d,D) = tf(t,d) \times idf(t,D) $$
يمكن أيضًا العثور على أفكار مشابهة لـ Bag of Words خارج المشكلات النصية، على سبيل المثال: حقيبة المواقع في [مسابقة Catch Me If You Can](https://inclass.kaggle.com/c/catch-me-if-you-can-intruder-detection-through-webpage-session-tracking)، [حقيبة التطبيقات](https://www.kaggle.com/xiaoml/talkingdata-mobile-user-demographics/bag-of-app-id-python-2-27392)، [حقيبة الأحداث](http://www.interdigital.com/download/58540a46e3b9659c9f000372)، وما إلى ذلك.
![صورة](../../img/bag_of_words.png)
باستخدام هذه الخوارزميات، من الممكن الحصول على حل عملي لمشكلة بسيطة، والذي يمكن أن يكون بمثابة خط الأساس. ومع ذلك، بالنسبة لأولئك الذين لا يحبون الكلاسيكيات، هناك أساليب جديدة. الطريقة الأكثر شيوعًا في الموجة الجديدة هي [Word2Vec](https://arxiv.org/pdf/1310.4546.pdf)، ولكن هناك بعض البدائل أيضًا ([GloVe](https://nlp.stanford.edu/pubs/glove.pdf)، [Fasttext](https://arxiv.org/abs/1607.01759)، وما إلى ذلك).يعد Word2Vec حالة خاصة من خوارزميات تضمين الكلمات. باستخدام Word2Vec والنماذج المشابهة، لا يمكننا فقط توجيه الكلمات في مساحة عالية الأبعاد (عادةً بضع مئات من الأبعاد) ولكن أيضًا مقارنة التشابه الدلالي بينها. هذا مثال كلاسيكي للعمليات التي يمكن إجراؤها على مفاهيم متجهة: الملك - الرجل + المرأة = الملكة.
![صورة](https://cdn-images-1.medium.com/max/800/1*K5X4N-MJKt8FGFtrTHwidg.gif)
ومن الجدير بالذكر أن هذا النموذج لا يفهم معنى الكلمات ولكنه يحاول ببساطة وضع المتجهات بحيث تكون الكلمات المستخدمة في السياق المشترك قريبة من بعضها البعض. إذا لم يؤخذ هذا في الاعتبار، فسوف يأتي الكثير من الأمثلة الممتعة.
تحتاج مثل هذه النماذج إلى التدريب على مجموعات بيانات كبيرة جدًا حتى تتمكن إحداثيات المتجهات من التقاط الدلالات. يمكن تنزيل نموذج مُدرب مسبقًا لمهامك الخاصة [هنا](https://github.com/3Top/word2vec-api#where-to-get-a-pretrained-models).
يتم تطبيق أساليب مماثلة في مجالات أخرى مثل المعلوماتية الحيوية. التطبيق غير المتوقع هو [food2vec](https://jaan.io/food2vec-augmented-cooking-machine-intelligence/). ربما يمكنك التفكير في بعض الأفكار الجديدة الأخرى؛ المفهوم عالمي بما فيه الكفاية.



### الصور
العمل مع الصور أسهل وأصعب في نفس الوقت. إنه أسهل لأنه من الممكن فقط استخدام إحدى الشبكات المشهورة المدربة مسبقًا دون الكثير من التفكير ولكنه أصعب لأنه إذا كنت بحاجة إلى التعمق في التفاصيل، فقد ينتهي بك الأمر إلى التعمق كثيرًا. لنبدأ من البداية.في الوقت الذي كانت فيه وحدات معالجة الرسومات أضعف ولم تحدث "نهضة الشبكات العصبية" بعد، كان توليد الميزات من الصور مجالًا معقدًا بحد ذاته. كان على المرء أن يعمل على مستوى منخفض، وتحديد الزوايا، وحدود المناطق، وإحصائيات توزيع الألوان، وما إلى ذلك. يمكن للمتخصصين ذوي الخبرة في رؤية الكمبيوتر رسم الكثير من أوجه التشابه بين الأساليب القديمة والشبكات العصبية؛ على وجه الخصوص، تشبه الطبقات التلافيفية في شبكات اليوم [سلسلة Haar](https://en.wikipedia.org/wiki/Haar-like_feature). إذا كنت مهتمًا بقراءة المزيد، فإليك بعض الروابط لبعض المكتبات المثيرة للاهتمام: [skimage](http://scikit-image.org/docs/stable/api/skimage.feature.html) و[SimpleCV](http://simplecv.readthedocs.io/en/latest/SimpleCV.Features.html).
في كثير من الأحيان، يتم استخدام شبكة عصبية تلافيفية للمشاكل المرتبطة بالصور. ليس عليك ابتكار البنية وتدريب الشبكة من الصفر. بدلاً من ذلك، قم بتنزيل شبكة حديثة مدربة مسبقًا مع الأوزان من المصادر العامة. غالبًا ما يقوم علماء البيانات بما يسمى الضبط الدقيق لتكييف هذه الشبكات مع احتياجاتهم عن طريق "فصل" آخر طبقات الشبكة المتصلة بالكامل، وإضافة طبقات جديدة مختارة لمهمة محددة، ثم تدريب الشبكة على البيانات الجديدة. إذا كانت مهمتك هي توجيه الصورة فقط (على سبيل المثال، استخدام بعض المصنفات غير المتصلة بالشبكة)، فأنت تحتاج فقط إلى إزالة الطبقات الأخيرة واستخدام الإخراج من الطبقات السابقة:


In [ ]:
# doesn't work with Python 3.7
# # Install Keras and tensorflow (https://keras.io/)
# from keras.applications.resnet50 import ResNet50, preprocess_input
# from keras.preprocessing import image
# from scipy.misc import face
# import numpy as np

# resnet_settings = {'include_top': False, 'weights': 'imagenet'}
# resnet = ResNet50(**resnet_settings)

# # What a cute raccoon!
# img = image.array_to_img(face())
# img

In [ ]:
# # In real life, you may need to pay more attention to resizing
# img = img.resize((224, 224))

# x = image.img_to_array(img)
# x = np.expand_dims(x, axis=0)
# x = preprocess_input(x)

# # Need an extra dimension because model is designed to work with an array
# # of images - i.e. tensor shaped (batch_size, width, height, n_channels)

# features = resnet.predict(x)


<img src='@@KEEP_00048@@ width=60%>
*إليك مصنفًا تم تدريبه على مجموعة بيانات واحدة وتكييفه مع مجموعة مختلفة عن طريق "فصل" الطبقة الأخيرة وإضافة طبقة جديدة بدلاً من ذلك.*ومع ذلك، لا ينبغي لنا أن نركز كثيرًا على تقنيات الشبكات العصبية. لا تزال الميزات التي تم إنشاؤها يدويًا مفيدة جدًا: على سبيل المثال، للتنبؤ بشعبية قائمة الإيجار، يمكننا أن نفترض أن الشقق المشرقة تجذب المزيد من الاهتمام وإنشاء ميزة مثل "متوسط ​​قيمة البكسل". يمكنك العثور على بعض الأمثلة الملهمة في وثائق [المكتبات ذات الصلة](http://pillow.readthedocs.io/en/3.1.x/reference/ImageStat.html).
إذا كان هناك نص على الصورة، فيمكنك قراءته دون كشف شبكة عصبية معقدة. على سبيل المثال، راجع [pytesseract](https://github.com/madmaze/pytesseract).



```python
import pytesseract
from PIL import Image
import requests
from io import BytesIO

##### Just a random picture from search
img = 'http://ohscurrent.org/wp-content/uploads/2015/09/domus-01-google.jpg'

img = requests.get(img)
img = Image.open(BytesIO(img.content))
text = pytesseract.image_to_string(img)

text

Out: 'Google'
```



يجب على المرء أن يفهم أن `pytesseract` ليس حلاً لكل شيء.



```python
##### This time we take a picture from Renthop
img = requests.get('https://photos.renthop.com/2/8393298_6acaf11f030217d05f3a5604b9a2f70f.jpg')
img = Image.open(BytesIO(img.content))
pytesseract.image_to_string(img)

Out: 'Cunveztible to 4}»'
```



هناك حالة أخرى لا تستطيع فيها الشبكات العصبية المساعدة وهي استخراج الميزات من المعلومات الوصفية. بالنسبة للصور، يقوم EXIF ​​بتخزين العديد من المعلومات الوصفية المفيدة: الشركة المصنعة وطراز الكاميرا، والدقة، واستخدام الفلاش، والإحداثيات الجغرافية للتصوير، والبرامج المستخدمة لمعالجة الصورة، والمزيد.



### البيانات الجغرافية المكانية
لا توجد البيانات الجغرافية في كثير من الأحيان في المشاكل، ولكن لا يزال من المفيد إتقان التقنيات الأساسية للعمل معها، خاصة وأن هناك عددا كبيرا من الحلول الجاهزة للاستخدام في هذا المجال.غالبًا ما يتم تقديم البيانات الجيومكانية على شكل عناوين أو إحداثيات (خط العرض، خط الطول). اعتمادًا على المهمة، قد تحتاج إلى عمليتين عكسيتين: التكويد الجغرافي (استعادة نقطة من عنوان) والتكويد الجغرافي العكسي (استعادة عنوان من نقطة). يمكن الوصول إلى كلتا العمليتين عمليًا عبر واجهات برمجة التطبيقات الخارجية من خرائط Google أو OpenStreetMap. تتميز أجهزة التشفير الجغرافي المختلفة بخصائصها الخاصة، وتختلف الجودة من منطقة إلى أخرى. لحسن الحظ، هناك مكتبات عالمية مثل [geopy](https://github.com/geopy/geopy) تعمل كمغلفات لهذه الخدمات الخارجية.
إذا كان لديك الكثير من البيانات، فسوف تصل بسرعة إلى حدود واجهة برمجة التطبيقات الخارجية. علاوة على ذلك، ليس من الأسرع دائمًا تلقي المعلومات عبر HTTP. ولذلك، فمن الضروري النظر في استخدام الإصدار المحلي من OpenStreetMap.
إذا كان لديك كمية صغيرة من البيانات، ووقتًا كافيًا، ولا ترغب في استخراج ميزات رائعة، فيمكنك استخدام `reverse_geocoder` بدلاً من OpenStreetMap:



```python
import reverse_geocoder as revgc

revgc.search((df.latitude, df.longitude))
Loading formatted geocoded file... 

Out: [OrderedDict([('lat', '40.74482'), 
                   ('lon', '-73.94875'), 
                   ('name', 'Long Island City'), 
                   ('admin1', 'New York'), 
                   ('admin2', 'Queens County'), 
                   ('cc', 'US')])]
```



عند العمل مع الترميز الجغرافي، يجب ألا ننسى أن العناوين قد تحتوي على أخطاء مطبعية، مما يجعل خطوة تنظيف البيانات ضرورية. تحتوي الإحداثيات على عدد أقل من الأخطاء المطبعية، ولكن يمكن أن يكون موقعها غير صحيح بسبب ضوضاء نظام تحديد المواقع العالمي (GPS) أو الدقة السيئة في أماكن مثل الأنفاق ومناطق وسط المدينة وما إلى ذلك. إذا كان مصدر البيانات جهازًا محمولاً، فقد لا يتم تحديد الموقع الجغرافي بواسطة نظام تحديد المواقع العالمي (GPS) ولكن عن طريق شبكات WiFi في المنطقة، مما يؤدي إلى ثغرات في الفضاء والنقل الآني. أثناء السفر في مانهاتن، يمكن أن يكون هناك فجأة موقع WiFi من شيكاغو.> يعتمد تتبع موقع WiFi على مزيج من عناوين SSID وعناوين MAC، والتي قد تتوافق مع نقاط مختلفة، على سبيل المثال. يقوم المزود الفيدرالي بتوحيد البرامج الثابتة لأجهزة التوجيه حتى عنوان MAC ووضعها في مدن مختلفة. حتى انتقال الشركة إلى مكتب آخر باستخدام أجهزة التوجيه الخاصة بها يمكن أن يسبب مشكلات.
تقع النقطة عادة بين البنية التحتية. هنا، يمكنك حقًا إطلاق العنان لخيالك وابتكار ميزات بناءً على خبرتك الحياتية ومعرفتك بالمجال: قرب نقطة ما من مترو الأنفاق، وعدد الطوابق في المبنى، والمسافة إلى أقرب متجر، وعدد أجهزة الصراف الآلي الموجودة حولك، وما إلى ذلك. ولأي مهمة، يمكنك بسهولة التوصل إلى العشرات من الميزات واستخراجها من مصادر خارجية مختلفة. بالنسبة للمشكلات خارج البيئة الحضرية، يمكنك التفكير في ميزات من مصادر أكثر تحديدًا، على سبيل المثال. الارتفاع عن مستوى سطح البحر.
إذا كانت نقطتان أو أكثر مترابطة، فقد يكون من المفيد استخراج الميزات من المسار بينهما. في هذه الحالة، ستكون المسافات (مسافة الدائرة الكبرى ومسافة الطريق المحسوبة بواسطة الرسم البياني للتوجيه)، وعدد المنعطفات مع نسبة المنعطفات من اليسار إلى اليمين، وعدد إشارات المرور، والتقاطعات، والجسور مفيدة. في إحدى مهامي الخاصة، قمت بإنشاء ميزة تسمى "تعقيد الطريق"، والتي حسبت المسافة المحسوبة بالرسم البياني مقسومة على GCD.



### التاريخ والوقت
قد تعتقد أن التاريخ والوقت موحدان بسبب شيوعهما، ولكن مع ذلك، لا تزال هناك بعض المخاطر.
لنبدأ بيوم الأسبوع، والذي يسهل تحويله إلى 7 متغيرات وهمية باستخدام تشفير واحد ساخن. بالإضافة إلى ذلك، سنقوم أيضًا بإنشاء ميزة ثنائية منفصلة لعطلة نهاية الأسبوع تسمى `is_weekend`.



```python
df['dow'] = df['created'].apply(lambda x: x.date().weekday())
df['is_weekend'] = df['created'].apply(lambda x: 1 if x.date().weekday() in (5, 6) else 0)
```


قد تتطلب بعض المهام ميزات تقويم إضافية. على سبيل المثال، يمكن ربط عمليات السحب النقدي بيوم الدفع؛ شراء بطاقة المترو، إلى بداية الشهر الجاري. بشكل عام، عند التعامل مع بيانات السلاسل الزمنية، من الجيد أن يكون لديك تقويم يتضمن أيام العطل الرسمية والأحوال الجوية غير الطبيعية والأحداث المهمة الأخرى.
> سؤال: ما هو العامل المشترك بين السنة الصينية الجديدة، وماراثون نيويورك، وتنصيب ترامب؟
> ج: يجب إدراجها جميعها في تقويم الحالات الشاذة المحتملة.
التعامل مع الساعة (الدقيقة، اليوم من الشهر...) ليس بالأمر السهل كما يبدو. إذا كنت تستخدم الساعة كمتغير حقيقي، فإننا نتعارض قليلاً مع طبيعة البيانات: `0<23` بينما `0:00:00 02.01> 01.01 23:00:00`. بالنسبة لبعض المشاكل، يمكن أن يكون هذا حاسما. وفي الوقت نفسه، إذا قمت بتشفيرها كمتغيرات فئوية، فسوف تقوم بتوليد عدد كبير من الميزات وتفقد معلومات حول القرب - سيكون الفرق بين 22 و23 هو نفس الفرق بين 22 و7.
توجد أيضًا بعض الأساليب الأكثر خصوصية لمثل هذه البيانات مثل إسقاط الوقت على دائرة واستخدام الإحداثيتين.


In [ ]:
def make_harmonic_features(value, period=24):
    value *= 2 * np.pi / period
    return np.cos(value), np.sin(value)


يحافظ هذا التحويل على المسافة بين النقاط، وهو أمر مهم للخوارزميات التي تقدر المسافة (kNN، SVM، k-means ...)


In [ ]:
from scipy.spatial import distance

euclidean(make_harmonic_features(23), make_harmonic_features(1))

In [ ]:
euclidean(make_harmonic_features(9), make_harmonic_features(11))

In [ ]:
euclidean(make_harmonic_features(9), make_harmonic_features(21))


ومع ذلك، فإن الفرق بين طرق الترميز هذه يصل إلى المنزلة العشرية الثالثة في المقياس.



### السلاسل الزمنية والويب وما إلى ذلك.
فيما يتعلق بالسلاسل الزمنية   —   لن نخوض في الكثير من التفاصيل هنا (غالبًا بسبب قلة خبرتي الشخصية)، لكنني سأوجهك إلى [مكتبة مفيدة تولد ميزات للسلاسل الزمنية تلقائيًا](https://github.com/blue-yonder/tsfresh).إذا كنت تعمل مع بيانات الويب، فعادةً ما يكون لديك معلومات حول وكيل المستخدم الخاص بالمستخدم. إنها ثروة من المعلومات. أولا، يحتاج المرء إلى استخراج نظام التشغيل منه. ثانيا، قم بعمل خاصية `is_mobile`. ثالثا، انظر إلى المتصفح.


In [ ]:
# Install pyyaml ua-parser user-agents
import user_agents

ua = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/56.0.2924.76 Chrome/56.0.2924.76 Safari/537.36"
ua = user_agents.parse(ua)

print("Is a bot? ", ua.is_bot)
print("Is mobile? ", ua.is_mobile)
print("Is PC? ", ua.is_pc)
print("OS Family: ", ua.os.family)
print("OS Version: ", ua.os.version)
print("Browser Family: ", ua.browser.family)
print("Browser Version: ", ua.browser.version)


> كما هو الحال في المجالات الأخرى، يمكنك التوصل إلى الميزات الخاصة بك بناءً على الحدس حول طبيعة البيانات. في وقت كتابة هذه السطور، كان Chromium 56 جديدًا، ولكن بعد مرور بعض الوقت، لن يحصل على هذا الإصدار سوى المستخدمين الذين لم يعيدوا تشغيل متصفحهم لفترة طويلة. في هذه الحالة، لماذا لا يتم تقديم ميزة تسمى "التأخر عن أحدث إصدار من المتصفح"؟
بالإضافة إلى نظام التشغيل والمتصفح، يمكنك الاطلاع على المُحيل (غير متوفر دائمًا)، [http_accept_language](https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers/Accept-Language)، والمعلومات التعريفية الأخرى.
المعلومة المفيدة التالية هي عنوان IP، الذي يمكنك من خلاله استخراج البلد وربما المدينة والمزود ونوع الاتصال (جوال/ثابت). عليك أن تفهم أن هناك مجموعة متنوعة من قواعد البيانات الوكيلة والقديمة، لذلك يمكن أن تحتوي هذه الميزة على ضوضاء. قد يحاول متخصصو إدارة الشبكة استخراج ميزات أكثر روعة مثل اقتراحات [استخدام VPN](https://habrahabr.ru/post/216295/). بالمناسبة، يتم دمج البيانات من عنوان IP جيدًا مع `http_accept_language`: إذا كان المستخدم جالسًا عند الوكلاء التشيليين وكانت لغة المتصفح هي `ru_RU`، فهناك شيء غير نظيف ويستحق البحث في العمود المقابل في الجدول (`is_traveler_or_proxy_user`).
تحتوي أي منطقة معينة على الكثير من التفاصيل التي يصعب على الفرد استيعابها بالكامل. لذلك أدعو الجميع لمشاركة تجاربهم ومناقشة استخراج الميزات وإنشائها في قسم التعليقات.



## تحويلات الميزة
### التطبيع وتغيير التوزيعيعد تحويل الميزات الرتيبة أمرًا بالغ الأهمية لبعض الخوارزميات وليس له أي تأثير على الخوارزميات الأخرى. وهذا هو أحد أسباب زيادة شعبية أشجار القرار وجميع الخوارزميات المشتقة منها (الغابات العشوائية وتعزيز التدرج). لا يستطيع الجميع أو لا يرغبون في التلاعب بالتحولات، وهذه الخوارزميات قوية في التعامل مع التوزيعات غير العادية.
هناك أيضًا أسباب هندسية بحتة: `np.log` هي طريقة للتعامل مع الأعداد الكبيرة التي لا تتناسب مع `np.float64`. وهذا استثناء وليس قاعدة؛ غالبًا ما يكون الدافع وراء ذلك هو الرغبة في تكييف مجموعة البيانات مع متطلبات الخوارزمية. تتطلب الطرق البارامترية عادةً الحد الأدنى من التوزيع المتماثل والأحادي للبيانات، وهو ما لا يتم تقديمه دائمًا في البيانات الحقيقية. قد تكون هناك متطلبات أكثر صرامة؛ تذكر [مقالتنا السابقة حول النماذج الخطية](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-4-linear-classification-and-regression-44a41b9b5220).
ومع ذلك، لا يتم فرض متطلبات البيانات فقط من خلال الأساليب البارامترية؛ [K أقرب الجيران](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-3-classification-decision-trees-and-k-nearest-neighbors-8613c6b6d2cd) سوف يتنبأ بالهراء الكامل إذا لم يتم تسوية الميزات، على سبيل المثال. عندما يقع أحد التوزيعين بالقرب من الصفر ولا يتجاوز (-1، 1) بينما يكون نطاق الآخر في حدود مئات الآلاف.
مثال بسيط: لنفترض أن المهمة هي التنبؤ بتكلفة الشقة من متغيرين   —   المسافة من وسط المدينة وعدد الغرف. ونادرا ما يتجاوز عدد الغرف 5 في حين أن المسافة من وسط المدينة يمكن أن تصل بسهولة إلى آلاف الأمتار.
أبسط تحويل هو القياس القياسي (أو تطبيع درجة Z):
$$ \large z= \frac{x-\mu}{\sigma} $$
لاحظ أن القياس القياسي لا يجعل التوزيع طبيعيًا بالمعنى الدقيق للكلمة.


In [ ]:
import numpy as np
from scipy.stats import beta, shapiro
from sklearn.preprocessing import StandardScaler

data = beta(1, 10).rvs(1000).reshape(-1, 1)
shapiro(data)

In [ ]:
# Value of the statistic, p-value
shapiro(StandardScaler().fit_transform(data))

# With such p-value we'd have to reject the null hypothesis of normality of the data


لكنه، إلى حد ما، يحمي من القيم المتطرفة:


In [ ]:
data = np.array([1, 1, 0, -1, 2, 1, 2, 3, -2, 4, 100]).reshape(-1, 1).astype(np.float64)
StandardScaler().fit_transform(data)

In [ ]:
(data - data.mean()) / data.std()

خيار آخر شائع إلى حد ما هو MinMax Scaling، والذي يجمع جميع النقاط ضمن فترة زمنية محددة مسبقًا (عادة (0، 1)).
$$ \large X_{norm}=\frac{X-X_{min}}{X_{max}-X_{min}} $$


In [ ]:
from sklearn.preprocessing import MinMaxScaler

MinMaxScaler().fit_transform(data)

In [ ]:
(data - data.min()) / (data.max() - data.min())


StandardScaling وMinMax Scaling لهما تطبيقات متشابهة وغالبًا ما يكونان قابلين للتبادل بشكل أو بآخر. ومع ذلك، إذا كانت الخوارزمية تتضمن حساب المسافات بين النقاط أو المتجهات، فإن الاختيار الافتراضي هو StandardScaling. لكن MinMax Scaling مفيد للتصور من خلال جلب الميزات ضمن الفاصل الزمني (0، 255).
إذا افترضنا أن بعض البيانات لا يتم توزيعها بشكل طبيعي ولكن يتم وصفها بواسطة [التوزيع الطبيعي للسجل](https://en.wikipedia.org/wiki/Log-normal_distribution)، فيمكن تحويلها بسهولة إلى توزيع عادي:


In [ ]:
from scipy.stats import lognorm

data = lognorm(s=1).rvs(1000)
shapiro(data)

In [ ]:
shapiro(np.log(data))


التوزيع اللوغاريتمي الطبيعي مناسب لوصف الرواتب، وأسعار الأوراق المالية، وسكان المناطق الحضرية، وعدد التعليقات على المقالات على الإنترنت، وما إلى ذلك. ومع ذلك، لتطبيق هذا الإجراء، ليس من الضروري أن يكون التوزيع الأساسي بالضرورة لوغاريتميًا طبيعيًا؛ يمكنك محاولة تطبيق هذا التحويل على أي توزيع بذيل أيمن ثقيل. علاوة على ذلك، يمكن للمرء أن يحاول استخدام تحويلات أخرى مماثلة، وصياغة فرضياته الخاصة حول كيفية تقريب التوزيع المتاح إلى الوضع الطبيعي. ومن أمثلة هذه التحويلات [تحويل Box-Cox](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.boxcox.html) (اللوغاريتم حالة خاصة لتحويل Box-Cox) أو [تحويل Yeo-Johnson](https://gist.github.com/mesgarpour/f24769cd186e2db853957b10ff6b7a95) (يوسع نطاق التطبيق على الأرقام السالبة). بالإضافة إلى ذلك، يمكنك أيضًا محاولة إضافة ثابت إلى الميزة  — `np.log (x + const)`.في الأمثلة المذكورة أعلاه، عملنا مع البيانات الاصطناعية واختبرنا الحالة الطبيعية بدقة باستخدام اختبار شابيرو ويلك. دعونا نحاول إلقاء نظرة على بعض البيانات الحقيقية واختبار الحالة الطبيعية باستخدام طريقة أقل رسمية  — [مخطط Q-Q](https://en.wikipedia.org/wiki/Q%E2%80%93Q_plot). بالنسبة للتوزيع الطبيعي، سيبدو كخط قطري سلس، ويجب أن تكون الشذوذات البصرية مفهومة بشكل بديهي.
![صورة](../../img/qq_lognorm.png)
مؤامرة QQ للتوزيع اللوغاريتمي الطبيعي
![صورة](../../img/qq_log.png)
مؤامرة Q-Q لنفس التوزيع بعد أخذ اللوغاريتم


In [ ]:
# Let's draw plots!
import statsmodels.api as sm

# Let's take the price feature from Renthop dataset and filter by hands the most extreme values for clarity

price = df.price[(df.price <= 20000) & (df.price > 500)]
price_log = np.log(price)

# A lot of gestures so that sklearn didn't shower us with warnings
price_mm = (
    MinMaxScaler()
    .fit_transform(price.values.reshape(-1, 1).astype(np.float64))
    .flatten()
)
price_z = (
    StandardScaler()
    .fit_transform(price.values.reshape(-1, 1).astype(np.float64))
    .flatten()
)


مؤامرة Q-Q للميزة الأولية


In [ ]:
sm.qqplot(price, loc=price.mean(), scale=price.std())


مؤامرة QQ بعد StandardScaler. الشكل لا يتغير


In [ ]:
sm.qqplot(price_z, loc=price_z.mean(), scale=price_z.std())


مؤامرة QQ بعد MinMaxScaler. الشكل لا يتغير


In [ ]:
sm.qqplot(price_mm, loc=price_mm.mean(), scale=price_mm.std())


مؤامرة Q-Q بعد أخذ اللوغاريتم. الأمور تتحسن!


In [ ]:
sm.qqplot(price_log, loc=price_log.mean(), scale=price_log.std())


دعونا نرى ما إذا كانت التحولات يمكن أن تساعد النموذج الحقيقي بطريقة أو بأخرى. لا توجد رصاصة فضية هنا.



### التفاعلات
إذا كانت التحولات السابقة تبدو وكأنها تعتمد على الرياضيات، فإن هذا الجزء يتعلق أكثر بطبيعة البيانات؛ يمكن أن يعزى إلى كل من تحويلات الميزات وإنشاء الميزات.
دعنا نعود مرة أخرى إلى مشكلة Two Sigma Connect: الاستعلامات عن قائمة الإيجار. ومن الميزات الموجودة في هذه المشكلة عدد الغرف والسعر. يشير المنطق إلى أن تكلفة الغرفة الفردية أكثر دلالة من التكلفة الإجمالية، حتى نتمكن من إنشاء مثل هذه الميزة.


In [ ]:
rooms = df["bedrooms"].apply(lambda x: max(x, 0.5))
# Avoid division by zero; .5 is chosen more or less arbitrarily
df["price_per_bedroom"] = df["price"] / rooms

يجب أن تحد نفسك في هذه العملية. إذا كان هناك عدد محدود من الميزات، فمن الممكن إنشاء جميع التفاعلات الممكنة ثم التخلص من التفاعلات غير الضرورية باستخدام التقنيات الموضحة في القسم التالي. بالإضافة إلى ذلك، ليس كل التفاعلات بين الميزات يجب أن يكون لها معنى مادي؛ على سبيل المثال، السمات متعددة الحدود (راجع [sklearn.preprocessing.PolynomialFeatures](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)) غالبًا ما تستخدم في النماذج الخطية ويكاد يكون من المستحيل تفسيرها.



### ملء القيم المفقودة
لا يمكن للعديد من الخوارزميات العمل مع القيم المفقودة، وغالبًا ما يوفر العالم الحقيقي بيانات بها فجوات. ولحسن الحظ، هذه إحدى المهام التي لا تحتاج إلى أي إبداع من أجلها. توفر كل من مكتبات بايثون الرئيسية لتحليل البيانات حلولاً سهلة الاستخدام: [pandas.DataFrame.fillna](http://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.fillna.html) و[sklearn.preprocessing.Imputer](http://scikit-learn.org/stable/modules/preprocessing.html#imputation).
هذه الحلول ليس لها أي سحر يحدث خلف الكواليس. تعد طرق التعامل مع القيم المفقودة واضحة جدًا:
* تشفير القيم المفقودة بقيمة فارغة منفصلة مثل `"n/a"` (للمتغيرات الفئوية)؛
* استخدم القيمة الأكثر احتمالاً للميزة (المتوسط ​​أو الوسيط للمتغيرات الرقمية، والقيمة الأكثر شيوعًا للمتغيرات الفئوية)؛
* أو على العكس من ذلك، التشفير باستخدام بعض القيمة المتطرفة (جيد لنماذج شجرة القرار لأنه يسمح للنموذج بإنشاء قسم بين القيم المفقودة وغير المفقودة)؛
* بالنسبة للبيانات المطلوبة (مثل السلاسل الزمنية)، خذ القيمة المجاورة  — التالي أو السابق.
![صورة](https://cdn-images-1.medium.com/max/800/0*Ps-v8F0fBgmnG36S.)تقترح حلول المكتبة سهلة الاستخدام أحيانًا الالتزام بشيء مثل `df = df.fillna(0)` وعدم التعرق على الفجوات. لكن هذا ليس الحل الأفضل: يستغرق إعداد البيانات وقتًا أطول من بناء النماذج، لذا فإن ملء الفجوات دون تفكير قد يؤدي إلى إخفاء خطأ في المعالجة وإتلاف النموذج.



## اختيار الميزة
لماذا قد يكون من الضروري تحديد الميزات؟ بالنسبة للبعض، قد تبدو هذه الفكرة غير بديهية، ولكن هناك سببين مهمين على الأقل للتخلص من الميزات غير المهمة. الأول واضح لكل مهندس: كلما زاد عدد البيانات، زاد التعقيد الحسابي. طالما أننا نعمل مع مجموعات بيانات الألعاب، فإن حجم البيانات لا يمثل مشكلة، ولكن بالنسبة لأنظمة الإنتاج المحملة الحقيقية، ستكون مئات الميزات الإضافية ملموسة تمامًا. السبب الثاني هو أن بعض الخوارزميات تأخذ الضوضاء (الميزات غير الإعلامية) كإشارة وتبالغ في التناسب.
### الأساليب الإحصائية
المرشح الأكثر وضوحًا للإزالة هو الميزة التي تظل قيمتها دون تغيير، أي أنها لا تحتوي على معلومات على الإطلاق. إذا بنينا على هذه الفكرة، فمن المعقول أن نقول إن الميزات ذات التباين المنخفض أسوأ من تلك ذات التباين العالي. لذلك، يمكن للمرء أن يفكر في قطع الميزات مع تباين أقل من عتبة معينة.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import VarianceThreshold

x_data_generated, y_data_generated = make_classification()
x_data_generated.shape

In [ ]:
VarianceThreshold(0.7).fit_transform(x_data_generated).shape

In [ ]:
VarianceThreshold(0.8).fit_transform(x_data_generated).shape

In [ ]:
VarianceThreshold(0.9).fit_transform(x_data_generated).shape


هناك طرق أخرى أيضًا [تعتمد على الإحصائيات الكلاسيكية](http://scikit-learn.org/stable/modules/feature_selection.html#univariate-feature-selection).


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

x_data_kbest = SelectKBest(f_classif, k=5).fit_transform(
    x_data_generated, y_data_generated
)
x_data_varth = VarianceThreshold(0.9).fit_transform(x_data_generated)

In [ ]:
logit = LogisticRegression(solver="lbfgs", random_state=17)

In [ ]:
cross_val_score(
    logit, x_data_generated, y_data_generated, scoring="neg_log_loss", cv=5
).mean()

In [ ]:
cross_val_score(
    logit, x_data_kbest, y_data_generated, scoring="neg_log_loss", cv=5
).mean()

In [ ]:
cross_val_score(
    logit, x_data_varth, y_data_generated, scoring="neg_log_loss", cv=5
).mean()


يمكننا أن نرى أن الميزات التي اخترناها قد حسنت جودة المصنف. وبطبيعة الحال، هذا المثال مصطنع بحتة؛ ومع ذلك، فإنه يستحق استخدام لمشاكل حقيقية.



### الاختيار عن طريق النمذجةهناك طريقة أخرى تتمثل في استخدام بعض النماذج الأساسية لتقييم الميزات لأن النموذج سيُظهر بوضوح أهمية الميزات. عادةً ما يتم استخدام نوعين من النماذج: بعض التركيبات "الخشبية" مثل [Random Forest](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-5-ensembles-of-algorithms-and-random-forest-8e05246cbba7) أو نموذج خطي مع تنظيم Lasso بحيث يكون عرضة لإبطال أوزان الميزات الضعيفة. المنطق بديهي: إذا كانت الميزات عديمة الفائدة بشكل واضح في نموذج بسيط، فليست هناك حاجة لسحبها إلى نموذج أكثر تعقيدًا.


In [ ]:
# Synthetic example

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

x_data_generated, y_data_generated = make_classification()

rf = RandomForestClassifier(n_estimators=10, random_state=17)
pipe = make_pipeline(SelectFromModel(estimator=rf), logit)

print(
    cross_val_score(
        logit, x_data_generated, y_data_generated, scoring="neg_log_loss", cv=5
    ).mean()
)
print(
    cross_val_score(
        rf, x_data_generated, y_data_generated, scoring="neg_log_loss", cv=5
    ).mean()
)
print(
    cross_val_score(
        pipe, x_data_generated, y_data_generated, scoring="neg_log_loss", cv=5
    ).mean()
)


يجب ألا ننسى أن هذه ليست حلاً سحريًا مرة أخرى، بل يمكن أن تجعل الأداء أسوأ.


In [ ]:
# x_data, y_data = get_data()
x_data = x_data_generated
y_data = y_data_generated

pipe1 = make_pipeline(StandardScaler(), SelectFromModel(estimator=rf), logit)

pipe2 = make_pipeline(StandardScaler(), logit)

print(
    "LR + selection: ",
    cross_val_score(pipe1, x_data, y_data, scoring="neg_log_loss", cv=5).mean(),
)
print(
    "LR: ", cross_val_score(pipe2, x_data, y_data, scoring="neg_log_loss", cv=5).mean()
)
print("RF: ", cross_val_score(rf, x_data, y_data, scoring="neg_log_loss", cv=5).mean())


### بحث الشبكة
أخيرًا، وصلنا إلى الطريقة الأكثر موثوقية، والتي تعد أيضًا الأكثر تعقيدًا من الناحية الحسابية: البحث الشبكي التافه. تدريب نموذج على مجموعة فرعية من الميزات، وتخزين النتائج، وتكرار ذلك لمجموعات فرعية مختلفة، ومقارنة جودة النماذج لتحديد أفضل مجموعة ميزات. يُطلق على هذا الأسلوب اسم [الاختيار الشامل للميزات](http://rasbt.github.io/mlxtend/user_guide/feature_selection/ExhaustiveFeatureSelector/).
عادةً ما يستغرق البحث في جميع المجموعات وقتًا طويلاً، لذا يمكنك محاولة تقليل مساحة البحث. قم بإصلاح عدد صغير من N، وقم بالتكرار عبر جميع مجموعات ميزات N، ثم اختر أفضل مجموعة، ثم قم بالتكرار عبر مجموعات ميزات (N + 1) بحيث يتم إصلاح أفضل مجموعة سابقة من الميزات ويتم أخذ ميزة جديدة واحدة فقط بعين الاعتبار. من الممكن التكرار حتى نصل إلى الحد الأقصى لعدد الخصائص أو حتى تتوقف جودة النموذج عن الزيادة بشكل ملحوظ. تسمى هذه الخوارزمية [الاختيار التسلسلي للميزات](http://rasbt.github.io/mlxtend/user_guide/feature_selection/SequentialFeatureSelector/).
يمكن عكس هذه الخوارزمية: البدء بمساحة الميزات الكاملة وإزالة الميزات واحدة تلو الأخرى حتى لا يؤثر ذلك على جودة النموذج أو حتى يتم الوصول إلى العدد المطلوب من الميزات.


In [ ]:
# Install mlxtend
from mlxtend.feature_selection import SequentialFeatureSelector

selector = SequentialFeatureSelector(
    logit, scoring="neg_log_loss", verbose=2, k_features=3, forward=False, n_jobs=-1
)

selector.fit(x_data, y_data)

ألقِ نظرة على كيفية تنفيذ هذا النهج في [نواة Kaggle بسيطة ولكنها أنيقة] (https://www.kaggle.com/arsenyinfo/easy-feature-selection-pipeline-0-55-at-lb).